# Level 2: cache architecture you can exercise

This notebook is a starting workspace for address decomposition, data selection,
direct-map conflicts and capacity experiments. It uses the real demand-only
software model in `cache_model.py`, not a picture-only placeholder.

The model is untimed: demand misses fill immediately. It has no prefetcher,
ready/valid control or simulated memory latency. Use Level 5 RTL replay for
performance. Original modelled matrix traces are not physical CPU traces.

![Phase 1 architecture](cache_architecture.svg)


In [1]:
from pathlib import Path
import json
import sys

# Run from repository root, Phase1, level2, or a child directory.
phase_candidates = [candidate
                    for base in (Path.cwd(), *Path.cwd().parents)
                    for candidate in (base, base / 'Phase1')]
PHASE1 = next((candidate for candidate in phase_candidates
               if (candidate / 'level2/cache_model.py').is_file()
               and (candidate / 'level4/rtl/ai_cache.sv').is_file()), None)
if PHASE1 is None:
    raise RuntimeError('Open this notebook inside the Micorn-day repository')
if str(PHASE1 / 'level2') not in sys.path:
    sys.path.insert(0, str(PHASE1 / 'level2'))
from cache_model import DirectMappedCache, DEMO_ADDRESSES, simulate, word_at
print('Loaded the Phase 1 cache model using portable repository discovery.')


Loaded the Phase 1 cache model using portable repository discovery.


## 1. Start with the actual Phase 1 geometry

The default has 64 direct-mapped lines, each 16 bytes. It holds 1,024 data bytes.
Four 32-bit words share each line. Metadata is additional storage; it is not part
of that 1 KiB data capacity.


In [2]:
cache = DirectMappedCache(index_bits=6)
print(json.dumps(cache.summary(), indent=2))


{
  "model": "untimed demand-only direct mapping; immediate demand refill",
  "index_bits": 6,
  "line_bytes": 16,
  "line_count": 64,
  "data_capacity_bytes": 1024,
  "accesses": 0,
  "hits": 0,
  "misses": 0,
  "demand_hit_rate": null,
  "demand_dram_reads": 0,
  "counter_semantics": "unbounded Python counts; hardware counters wrap at 32 bits",
  "timing_or_prefetch_modelled": false
}


## 2. Decode an address before simulating it

Byte address `0x1234` should select line `0x123`, index 35, word 1 and refill
address `0x1230`. The conventional minimum tag is 4. The RTL stores the full
line identity, which includes redundant index bits.


In [3]:
mapping = cache.decode(0x1234)
print(json.dumps(mapping, indent=2))
assert mapping['line_identity'] == 0x123
assert mapping['index'] == 35
assert mapping['word_offset'] == 1
assert mapping['aligned_line_address'] == 0x1230


{
  "address": 4660,
  "line_identity": 291,
  "index": 35,
  "conventional_tag": 4,
  "word_offset": 1,
  "aligned_line_address": 4656
}


## 3. Watch hits and evictions

The first three addresses share a line. `0x1400` later conflicts with `0x1000`
at index zero. The expected result is seven accesses, three hits and four misses.
Change the addresses and explain the result before running the cell again.


In [4]:
demo = simulate(DEMO_ADDRESSES)
print('Address      Index Word Hit   Evicted line')
for event in demo['events']:
    victim = event['evicted_line_identity']
    print(f"0x{event['address']:08x}  {event['index']:5} {event['word_offset']:4} {str(event['hit']):5} "
          f"{'-' if victim is None else hex(victim)}")
print(json.dumps(demo['summary'], indent=2))
assert (demo['summary']['accesses'], demo['summary']['hits'], demo['summary']['misses']) == (7, 3, 4)


Address      Index Word Hit   Evicted line
0x00001000      0    0 False -
0x00001004      0    1 True  -
0x0000100c      0    3 True  -
0x00001010      1    0 False -
0x00001000      0    0 True  -
0x00001400      0    0 False 0x100
0x00001000      0    0 False 0x140
{
  "model": "untimed demand-only direct mapping; immediate demand refill",
  "index_bits": 6,
  "line_bytes": 16,
  "line_count": 64,
  "data_capacity_bytes": 1024,
  "accesses": 7,
  "hits": 3,
  "misses": 4,
  "demand_hit_rate": 0.42857142857142855,
  "demand_dram_reads": 4,
  "counter_semantics": "unbounded Python counts; hardware counters wrap at 32 bits",
  "timing_or_prefetch_modelled": false
}


## 4. Verify returned words, not just hit counts

The deterministic backing-memory word is `(address >> 2) XOR 0xcafe1234`.
One line fill stores four words; a hit selects the requested stored word.
This checks the same data convention as the RTL transport tests. It does not
execute matrix multiplication or represent application values.


In [5]:
word_cache = DirectMappedCache()
word_events = [word_cache.read(0x2000 + offset) for offset in (0, 4, 8, 12)]
for event in word_events:
    expected = word_at(event['address'])
    assert event['data'] == expected
    print(f"address=0x{event['address']:08x} data=0x{event['data']:08x} hit={event['hit']}")
assert [event['hit'] for event in word_events] == [False, True, True, True]


address=0x00002000 data=0xcafe1a34 hit=False
address=0x00002004 data=0xcafe1a35 hit=True
address=0x00002008 data=0xcafe1a36 hit=True
address=0x0000200c data=0xcafe1a37 hit=True


## 5. Compare capacities while keeping the line size fixed

Scan 96 lines twice. The touched-line footprint is 1,536 bytes, but footprint
alone does not determine every cache miss: mapping and access order matter.
The following is a demand-only count experiment, not a timing speedup study.


In [6]:
addresses = [0x1000 + 16 * (i % 96) for i in range(192)]
print('Data bytes  Lines  Hits  Misses  Demand hit rate')
for bits in (4, 6, 8):
    summary = simulate(addresses, index_bits=bits)['summary']
    print(f"{summary['data_capacity_bytes']:10} {summary['line_count']:6} "
          f"{summary['hits']:5} {summary['misses']:7} {summary['demand_hit_rate']:.2%}")


Data bytes  Lines  Hits  Misses  Demand hit rate
       256     16     0     192 0.00%
      1024     64    32     160 16.67%
      4096    256    96      96 50.00%


## Work to do next

1. Change the address pair so that it either shares an index or occupies distinct indices.
2. Compare 16-, 64- and 256-line capacities with the same demand sequence.
3. Run `PYTHONPATH=level2 python3 -m unittest discover -s level2/tests -v` from `Phase1/`.
   The regression regenerates all 13 Level 5 traces and compares no-prefetch
   model counts with recorded RTL counts.
4. Read [cache_specification.md](cache_specification.md) before changing request
   handling, line size, associativity or the RTL data path.
5. Continue to the [prefetch strategy](../level3/README.md). A prefetcher changes
   residency and timing, so this demand-only model is no longer sufficient.

No phase-two buffer, adaptive feedback or new measured speedup is implemented here.
